# 旅さき: GPU での実測（#435 Guided Decoding と Gemma 4 E4B）

**使いかた**: 上のメニューの「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選び、「ランタイム」→「**すべてのセルを実行**」を押すだけです。
20〜40分ほどで、いちばん下のセルに **貼り付ける結果** が出ます。それをそのままチャットに貼ってください。

やること:

1. Node.js 22 と Ollama を入れ、Gemma 4 E4B（`gemma4:e4b`）を落とします（ログインも鍵も要りません）
2. `tools/tool_call_eval.mjs` で、道具の呼び出しの崩れを **しばりなし（free）／JSON Schema（schema）／tools** で比べます
3. `tools/model_eval.mjs` で、元の Gemma 4 E4B を採点します（チューニング前の基準の数字）
4. （任意）vLLM でも同じ比べを回します。既定では回しません（下の `USE_VLLM`）

お金はかかりません（Colab の無料枠）。途中で止まったら、もう一度「すべてのセルを実行」を押してください。

In [ ]:
# ── 設定（ふつうは変えなくて大丈夫です）──
MODELS = ["gemma4:e4b"]          # Ollama のモデル名。比べたければ "gemma4:e2b" も足せます
REPEAT = 1                       # 同じ問いを何回聞くか（2 以上にすると揺れも見えます）
TEMPERATURE = 0                  # 0 = いつも同じ答え。揺れを見るなら 0.7
EVAL_LIMIT = 60                  # model_eval で採点する問題の数（0 = 全部）
USE_VLLM = False                 # True にすると vLLM でも比べます（Hugging Face のトークンが要ります）
VLLM_MODEL = "google/gemma-3-4b-it"  # vLLM で落とすモデル（Hugging Face で利用条件に同意しておく）
REPO = "https://github.com/shitianliang1000-alt/tabisaki-app.git"
BRANCHES = ["main", "claude/project-thread-5hlkzs"]  # 先にあるほうを使います（まとめの PR が入る前でも動くように）

In [ ]:
# GPU の確認
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    print("⚠️ GPU がありません。「ランタイム」→「ランタイムのタイプを変更」で T4 GPU を選んでから、もう一度すべて実行してください。")

In [ ]:
%%bash
# Node.js 22 を入れます
set -e
if ! node --version 2>/dev/null | grep -q '^v2[2-9]'; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null
  apt-get install -y nodejs > /dev/null
fi
node --version

In [ ]:
# リポジトリを落とします（tools/tool_call_eval.mjs があるブランチを使います）
import os, subprocess
os.chdir("/content")
subprocess.run(["rm", "-rf", "tabisaki"])
used = None
for b in BRANCHES:
    r = subprocess.run(["git", "clone", "-q", "--depth=1", "-b", b, REPO, "tabisaki"], capture_output=True, text=True)
    if r.returncode == 0 and os.path.exists("tabisaki/tools/tool_call_eval.mjs"):
        used = b
        break
    subprocess.run(["rm", "-rf", "tabisaki"])
if not used:
    raise SystemExit("リポジトリを落とせませんでした。ネットにつながっているか確かめてください。")
os.chdir("/content/tabisaki")
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("ブランチ:", used, " コミット:", COMMIT)

In [ ]:
%%bash
# Ollama を入れて、裏で動かします
set -e
if ! command -v ollama > /dev/null; then
  apt-get install -y zstd > /dev/null 2>&1 || true
  curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
fi
pkill -f "ollama serve" 2>/dev/null || true
OLLAMA_CONTEXT_LENGTH=8192 OLLAMA_KEEP_ALIVE=1h nohup ollama serve > /content/ollama.log 2>&1 &
for i in $(seq 1 60); do curl -s localhost:11434/api/version > /dev/null && break; sleep 1; done
curl -s localhost:11434/api/version; echo

In [ ]:
# モデルを落とします（E4B は 10GB 前後。数分かかります）
import subprocess
for m in MODELS:
    print("落としています:", m)
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print(r.stdout[-300:] or r.stderr[-300:])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

In [ ]:
# 1. 道具の呼び出しの崩れを比べます（free / schema / tools）
import subprocess, json
RESULTS = {}
for m in MODELS:
    out = f"/content/tool-call-{m.replace(':','-')}.json"
    cmd = ["node", "tools/tool_call_eval.mjs", "--url=http://localhost:11434", f"--model={m}",
           "--modes=free,schema,tools", f"--repeat={REPEAT}", f"--temperature={TEMPERATURE}", f"--out={out}"]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
    RESULTS[("ollama", m)] = r.stdout.strip()

In [ ]:
# 2. 元の Gemma を採点します（チューニング前の基準）
import subprocess, os
r = subprocess.run(["node", "tools/export_training_data.mjs", "--out=/content/training"], capture_output=True, text=True)
print(r.stdout[-800:] or r.stderr[-800:])
EVALS = {}
for m in MODELS:
    out = f"/content/model-eval-{m.replace(':','-')}.json"
    cmd = ["node", "tools/model_eval.mjs", "--url=http://localhost:11434", f"--model={m}",
           "--data=/content/training/eval.jsonl", f"--limit={EVAL_LIMIT}", f"--out={out}"]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
    EVALS[m] = r.stdout.strip()

In [ ]:
# 3.（任意）vLLM でも比べます。USE_VLLM = True のときだけ動きます。
# Hugging Face のトークンを、左の鍵のマーク（シークレット）に HF_TOKEN という名前で入れておいてください。
import subprocess, time, os
if USE_VLLM:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN が読めませんでした:", e)
    subprocess.run(["ollama", "stop", MODELS[0]], capture_output=True)  # GPU のメモリを空けます
    subprocess.run(["pkill", "-f", "ollama serve"])
    subprocess.run(["pip", "install", "-q", "vllm"])
    log = open("/content/vllm.log", "w")
    vllm = subprocess.Popen(["vllm", "serve", VLLM_MODEL, "--port", "8000", "--dtype", "half",
                             "--max-model-len", "4096", "--gpu-memory-utilization", "0.9"],
                            stdout=log, stderr=subprocess.STDOUT)
    for i in range(900):
        if subprocess.run(["curl", "-s", "localhost:8000/v1/models"], capture_output=True).returncode == 0:
            break
        time.sleep(2)
    for style in ["json_schema", "guided_json"]:
        cmd = ["node", "tools/tool_call_eval.mjs", "--url=http://localhost:8000", f"--model={VLLM_MODEL}",
               "--modes=free,schema,tools", f"--style={style}", f"--repeat={REPEAT}", f"--temperature={TEMPERATURE}",
               f"--out=/content/tool-call-vllm-{style}.json"]
        print("$", " ".join(cmd))
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(r.stdout[-4000:] or r.stderr[-2000:])
        RESULTS[(f"vllm/{style}", VLLM_MODEL)] = r.stdout.strip()
        if r.returncode == 0:
            break
    vllm.terminate()
else:
    print("vLLM は回しません（USE_VLLM = False）。")

In [ ]:
# 4. 貼り付ける結果（このセルの出力を、まるごとチャットに貼ってください）
import subprocess, datetime
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("==== ここから貼り付け ====")
print(f"## GPU での実測（{datetime.date.today().isoformat()}、{gpu}、コミット {COMMIT}）")
for (server, m), table in RESULTS.items():
    print(f"\n### tool_call_eval: {server} / {m}（repeat={REPEAT}, temperature={TEMPERATURE}）\n")
    print(table or "（結果なし。上のセルのエラーを見てください）")
for m, summary in EVALS.items():
    print(f"\n### model_eval: {m}（{EVAL_LIMIT or '全'}問）\n")
    print("```json")
    print(summary or "（結果なし）")
    print("```")
print("\n==== ここまで ====")